# Feature Selection & Engineering
1. Initial feature selection based on EDA and business understanding
2. Outlier treatment (fit in train)
3. Create ratios
4. Scaling
5. Feature Engineering and Missing Value Treatment
6. Categorical encoding

Split → fit scaler en X_train → transform X_train y X_test

## 1. Initial feature selection based on EDA and business understanding
The objective is to retain variables that:

1. Are **available at application time**,
2. Have **economic meaning** in a credit risk context,
3. Show **directional relationship with default**, even if weak,
4. Avoid unnecessary noise, leakage, or instability given the rare-event setting.


### Scope and Exclusions

Only variables observable **before loan approval** are considered. As a result:

**Explicitly excluded variables include:**
* Loan status refinements after origination
* Payment history variables observed after the loan was issued
* Any outcome-driven or time-forward information

This preserves a realistic credit-scoring setup and avoids data leakage.


### Core Feature Groups

Feature selection is organised by **economic interpretation**, not just statistical correlation.


#### Borrower Financial Capacity

These variables describe the borrower’s ability to service debt.

**Retained features**
* `annual_income`
* `debt_to_income`
* `homeownership`

**Rationale**
* Income captures earning capacity but is skewed → requires transformation.
* DTI directly measures financial stress and shows weak but consistent directional signal.
* Homeownership differentiates fixed obligation structures (rent vs mortgage vs own).


#### Employment Stability

**Retained features**
* `emp_length`
* `verified_income`

**Rationale**
* Employment length shows heterogeneity but contributes stability context.
* Income verification adds reliability information beyond raw income.


#### Credit History Length and Structure

**Retained features**
* `earliest_credit_line`
* `total_credit_lines`
* `open_credit_lines`

**Rationale**
* Credit history length captures borrower maturity but does not imply lower risk on its own.
* Number of credit lines reflects exposure and complexity of obligations.
* These variables show moderate differences between default and non-default groups and complement leverage metrics.


#### Past Delinquency and Adverse Credit Events

These features are economically intuitive but **require careful selection due to sparsity**.

**Retained features**
* `months_since_last_delinq`
* `public_record_bankrupt`

**Explicitly excluded features**
* `num_historical_failed_to_pay`
* `delinq_2yr`
* `months_since_90d_late`

**Rationale**

* `num_historical_failed_to_pay` shows:
  * Heavy concentration at zero
  * Strong overlap between defaulted and non-defaulted borrowers
  * Unstable behaviour given only 7 default observations
    Its weak positive correlation with default reflects **directional intuition**, but not robust standalone signal.

* `delinq_2yr` exhibits:
  * Median equal to zero for both classes
  * Near-zero variance among defaulters
  * Counterintuitive mean behaviour (lower for defaulters), strongly suggesting **small-sample artifacts**

* `months_since_90d_late` shows:
  * Almost no variability in the default class
  * Collapse to a single value for defaulters

> In a rare-event setting, variables with sparse distributions and unstable class behaviour introduce noise and reduce generalisation performance.
> These features were therefore excluded in favour of more stable credit history proxies.


#### Legal and Tax Signals

**Retained features**
* `tax_liens`

**Rationale**
* Despite low frequency, tax liens show the **strongest categorical default rate differences**.
* Economically intuitive red-flag variable, retained with regularisation to control variance.


#### Loan Characteristics

**Retained features**
* `loan_amount`
* `loan_purpose`
* `application_type`

**Rationale**
* Loan purpose captures intent and risk profile (e.g. housing-related loans show higher default rates).
* Joint vs individual applications show minimal difference but are retained for completeness.
* Loan amount reflects exposure and interacts with income and leverage.


### Variables Excluded from Modeling

The following variables are excluded due to **low signal, instability, or redundancy**:
* Sparse delinquency counters with overlapping class distributions
* Variables with near-zero variance among defaulters
* Post-origination payment variables
* Highly granular categorical levels with insufficient observations
* Raw versions of variables replaced by transformed equivalents (e.g. unscaled income)

This helps reduce noise and overfitting in an already imbalanced dataset.


### Feature Transformation Strategy

Based on EDA findings:

* **Log-transform or cap**

  * `annual_income`
  * credit line counts
* **Scale robustly**

  * DTI and ratio-based features
* **Encode categoricals**

  * One-hot encoding with rare-category grouping
* **Preserve monotonicity where possible**

  * Useful for explainability in logistic or tree-based models

### 5. Modeling Philosophy

Feature selection is intentionally **conservative**:

* Preference is given to **economically interpretable and stable variables**
* Weak univariate signals are discarded when they are:
  * Sparse
  * Unstable
  * Likely driven by sampling noise

Given the **~1.5% default rate**, model performance will rely more on:

* Feature interactions
* Regularisation
* Threshold optimisation

rather than on any single dominant predictor.

In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv('loans_full_schema.csv')
# df_credit = df[df["loan_status"].isin(["Charged Off", "Fully Paid"])]
df_credit = df.loc[
    df["loan_status"].isin(["Charged Off", "Fully Paid"])
].copy()

target = "loan_status"
df_credit["loan_status"] = (df_credit["loan_status"] == "Charged Off").astype(int)


# Selected features based on EDA + credit intuition
num_features = [
    # Financial capacity
    "annual_income",
    "annual_income_joint",
    "debt_to_income",

    
    # Employment stability
    "emp_length",
    
    # Credit history & structure
    "earliest_credit_line",
    "total_credit_lines",
    "open_credit_lines",
    "total_debit_limit",
    "num_total_cc_accounts",
    "num_open_cc_accounts",

    # Past delinquency / adverse events
    "months_since_last_delinq",
    
    # Loan characteristics
    "loan_amount",
    
]

cat_features = [
    "homeownership",
    "verified_income",
    "public_record_bankrupt",
    "tax_liens",
    "loan_purpose",
    "application_type",
    "state",
    "emp_title"
]

features = num_features + cat_features

df_model = df_credit[features + ["loan_status"]].copy()
print(f"Modeling dataset shape: {df_model.shape}")
df_model.head()

Modeling dataset shape: (454, 21)


,annual_income,annual_income_joint,debt_to_income,emp_length,earliest_credit_line,total_credit_lines,open_credit_lines,total_debit_limit,num_total_cc_accounts,num_open_cc_accounts,months_since_last_delinq,loan_amount,homeownership,verified_income,public_record_bankrupt,tax_liens,loan_purpose,application_type,state,emp_title,loan_status
18,210000.0,NaN,9.53,10.0,2003,18,7,17500,6,2,NaN,5000,MORTGAGE,Verified,0,0,medical,individual,IL,operational risk manager,0
19,83000.0,NaN,18.44,1.0,2005,11,6,8300,7,4,14.0,20000,MORTGAGE,Source Verified,0,0,debt_consolidation,individual,CA,welder,0
34,140000.0,NaN,13.82,10.0,1993,21,12,58100,13,9,NaN,15000,MORTGAGE,Not Verified,0,0,debt_consolidation,individual,CA,deputy,0
35,70000.0,NaN,0.00,1.0,2004,13,3,0,11,3,31.0,2400,OWN,Source Verified,0,0,small_business,individual,MD,armed protection officer,0
107,44000.0,NaN,24.77,2.0,2011,16,9,9500,8,6,NaN,7200,MORTGAGE,Verified,0,0,home_improvement,individual,FL,crew chief,0


In [10]:
# Anything that is “learned” from the data comes AFTER the split.
from sklearn.model_selection import train_test_split

vars_to_scale = [
    "annual_income",
    "annual_income_joint",
    "emp_length",
    "earliest_credit_line",
    "total_credit_lines",
    "open_credit_lines",
    "total_debit_limit",
    "num_total_cc_accounts",
    "num_open_cc_accounts",
    "months_since_last_delinq",
    "loan_amount",
    # Key: debt_to_income out of the scaler because it is a ratio
    "debt_to_income"
]

X = df_credit[vars_to_scale+cat_features].copy()
y = df_credit[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42, stratify=y
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)
print("Default rate train:", y_train.mean(), "Default rate test:", y_test.mean())



Train shape: (363, 20) Test shape: (91, 20)
Default rate train: 0.01652892561983471 Default rate test: 0.01098901098901099


## 2. Outlier treatment (risk-aware)
Outlier treatment was considered but not applied. Exploratory analysis showed that numerical variables do not present extreme or implausible values, and the dataset size is relatively small. Since extreme observations may carry genuine credit risk information, removing or capping them could distort the underlying risk signal. Instead, scaling and model regularisation are relied upon to ensure numerical stability.

If severe outliers had been detected, a risk-aware winsorization strategy would have been used, fitting percentile-based caps on the training set only and applying them consistently to the test set.

> **Winsorization** (or *capping*) caps extreme values at predefined thresholds instead of removing observations. It is commonly used in credit risk when outliers are plausible but dominate the scale. Percentile-based caps are fitted on the training set only to prevent data leakage.

```python
import numpy as np

def fit_caps(X, cols, lower_q=0.01, upper_q=0.99):
    caps = {}
    for col in cols:
        lower = X[col].quantile(lower_q)
        upper = X[col].quantile(upper_q)
        caps[col] = (lower, upper)
    return caps

def apply_caps(X, caps):
    X_capped = X.copy()
    for col, (lower, upper) in caps.items():
        X_capped[col] = X_capped[col].clip(lower, upper)
    return X_capped

# Example usage (ONLY after train/test split)
num_cols = [
    "annual_income",
    "loan_amount",
    "total_debit_limit"
]

caps = fit_caps(X_train, num_cols)      # fit ONLY on train
X_train_capped = apply_caps(X_train, caps)
X_test_capped  = apply_caps(X_test, caps)

```

## 3. Creation of financial risk ratios

A single financial risk ratio was created: the ratio between loan amount and annual income. This variable captures the relative burden of the requested loan compared to the borrower’s earning capacity and reflects the marginal risk introduced by the new exposure. Given the dataset size and the presence of existing leverage indicators, no additional ratios were engineered to avoid redundancy and overfitting.

In [11]:
print("Train shape before ratio:", X_train.shape)
print("Test shape before ratio:", X_test.shape)

# Creation of loan burden ratio
def compute_loan_income_ratio(df):
    income = df["annual_income"].copy()
    mask = (income.isna()) | (income <= 0)
    income[mask] = df.loc[mask, "annual_income_joint"]

    ratio = df["loan_amount"] / income
    return ratio

# Create ratio
X_train["loan_to_income_ratio"] = compute_loan_income_ratio(X_train)
X_test["loan_to_income_ratio"]  = compute_loan_income_ratio(X_test)

# Drop rows where ratio could not be computed
train_mask = X_train["loan_to_income_ratio"].notna()
test_mask  = X_test["loan_to_income_ratio"].notna()

X_train = X_train.loc[train_mask].copy()
y_train = y_train.loc[train_mask].copy()

X_test  = X_test.loc[test_mask].copy()
y_test  = y_test.loc[test_mask].copy()

print("Train shape after ratio:", X_train.shape)
print("Test shape after ratio:", X_test.shape)

Train shape before ratio: (363, 20)
Test shape before ratio: (91, 20)
Train shape after ratio: (363, 21)
Test shape after ratio: (91, 21)


In [13]:
vars_to_scale.append("loan_to_income_ratio")
vars_not_scale = ["debt_to_income"]

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# ColumnTransformer: scale only selected numeric columns
scaler_num_var = ColumnTransformer(
    transformers=[
        ("vars_to_scale",StandardScaler(),vars_to_scale),
        ("num_passthrough","passthrough", vars_not_scale),
        ("cat_passthrough", "passthrough", cat_features)
    ]
)

# Fit ONLY on train, then transform both train and test
X_train_scaled = scaler_num_var.fit_transform(X_train)
X_test_scaled = scaler_num_var.transform(X_test) 

"""
All preprocessing steps that estimate data-dependent parameters are fitted exclusively
on the training set and then applied to the test set to prevent information leakage.
"""

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)



X_train_scaled shape: (363, 23)
X_test_scaled shape: (91, 23)


## 5. Feature Scaling and Normalisation

To ensure numerical stability and comparable feature ranges, it was applied **standardisation (z-score scaling)** to continuous numerical variables with heterogeneous magnitudes.

### Scaling decisions
- **Scaled (StandardScaler):** all unbounded / wide-range numerical variables, including the engineered `loan_to_income_ratio`.
- **Not scaled:** `debt_to_income`, as it is a bounded and directly interpretable ratio.
- **Categorical variables:** passed through unchanged at this stage (they will be encoded in the next step).

### Leakage prevention
Scaling parameters (mean and standard deviation) are **fitted only on the training set** and then applied to both training and test sets:
- `fit_transform()` on `X_train`
- `transform()` on `X_test`

In [ ]:
from sklearn.impute import SimpleImputer

# Numeric + categorical imputers
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

# Copy to avoid SettingWithCopy issues
X_train_imp = X_train.copy()
X_test_imp  = X_test.copy()

# Fit on TRAIN only

used_numeric = vars_to_scale + vars_not_scale # Impute ALL numeric columns used by the preprocess (scaled + passthrough)

X_train_imp[used_numeric] = num_imputer.fit_transform(X_train_imp[used_numeric])
X_test_imp[used_numeric]  = num_imputer.transform(X_test_imp[used_numeric])

X_train_imp[cat_features] = cat_imputer.fit_transform(X_train_imp[cat_features])
X_test_imp[cat_features]  = cat_imputer.transform(X_test_imp[cat_features])

## 5. Feature Engineering and Missing Value Treatment

Prior to scaling and encoding, missing values were handled explicitly as part of the feature engineering process.

All numerical variables used in the preprocessing stage (both scaled and non-scaled features) were imputed using **median imputation**, fitted exclusively on the training set. This approach is robust to skewed financial distributions and reduces the influence of extreme values.

Categorical variables were imputed using the **most frequent category**, ensuring that missing values did not propagate into the encoding stage.

All imputation parameters were estimated on the training set only and subsequently applied to the test set, preventing information leakage and ensuring consistent preprocessing across datasets.


In [ ]:
from sklearn.preprocessing import OneHotEncoder

# ColumnTransformer: scale only selected numeric columns
preprocess = ColumnTransformer(
    transformers=[
        ("vars_to_scale",StandardScaler(),vars_to_scale),
        ("num_passthrough","passthrough", vars_not_scale),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
    ],
    remainder="drop"
)

X_train_preprocessed = preprocess.fit_transform(X_train)
X_test_preprocessed = preprocess.transform(X_test) 

print("Train shape after preprocess :", X_train_preprocessed.shape)
print("Test shape after preprocess:", X_test_preprocessed.shape)

Train shape after preprocess : (363, 377)
Test shape after preprocess: (91, 377)


## 5. Categorical Encoding

Categorical variables are converted into numeric features using **One-Hot Encoding**, which creates binary indicators for each category without imposing any ordinal relationship.

### Key implementation choices
- **OneHotEncoder(handle_unknown="ignore")** is used to ensure the pipeline remains robust if unseen categories appear in the test set.
- Categorical encoding is performed within a **ColumnTransformer**, alongside numerical preprocessing, to guarantee consistent transformations across datasets.

### Leakage prevention
The encoder is **fitted only on the training set** (`fit_transform` on `X_train`) and then applied to the test set (`transform` on `X_test`).  
This ensures that category levels from the test set do not influence the learned feature space.

### Result
After preprocessing, the feature space expands from the original 21 variables to 377 encoded features, reflecting the full one-hot representation of categorical inputs.
